## Transferir o notebook

In [8]:
%pip install -r requirements.txt

import kagglehub
path = kagglehub.dataset_download( "dansbecker/melbourne-housing-snapshot",                                                                   output_dir="data/melbourne",)

print("Ficheiro do dataset em:", path)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Ficheiro do dataset em: data/melbourne


## Objetivo

Ver que variáveis fazem uma venda ficar barata, normal ou cara em relação ao preço esperado. Usa o `melb_clean.csv` gerado pelo `01_EDA`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

from libs.decisiontrees import DecisionTreeClassifier

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 250)

SEMENTE = 42
CLASSES = ["baixo", "normal", "alto"]

O preço esperado de cada venda foi estimado com uma regressão Ridge sobre o log do preço, com `Rooms`, `Bathroom`, `Car`, `Distance`, `Landsize`, `Type`, `Suburb` e o mês da venda. `Landsize` e `Distance` entram como curvas separadas por tipo de imóvel. Mais de 12% abaixo do esperado é `baixo`, mais de 12% acima é `alto` e o resto é `normal`.

## Preparar os dados

Só as vendas concluídas têm classe, por isso removemos as linhas com `PI` e `VB` 

In [2]:
dados = pd.read_csv("data/melbourne/melb_clean.csv", parse_dates=["Date"])

# Remover PI e VB, porque só as vendas concluídas (S, SP, SA) têm classe
vendas = dados[dados["ClassePreco"].notna()].reset_index(drop=True)
print(f"{len(vendas)} vendas com classe (de {len(dados)} linhas)")
vendas["ClassePreco"].value_counts(normalize=True).round(3)

10817 vendas com classe (de 13579 linhas)


ClassePreco
normal    0.523
alto      0.247
baixo     0.230
Name: proportion, dtype: float64

Além das variáveis da regressão entram `BuildingArea`, `YearBuilt`, `Method`, `Regionname` e três variáveis novas: `AreaPorDivisao`, `Meses` desde a primeira venda e `Agencia`, com as 15 agências com mais vendas e as restantes em `outras`.

Removemos `Price`, `PrecoEsperado` e `DesvioPreco`, porque definem a classe, e as outras colunas de localização, porque já estão no preço esperado.

In [3]:
vendas["Meses"] = (vendas["Date"] - vendas["Date"].min()).dt.days / 30.44

principais = vendas["SellerG"].value_counts().index[:15]
vendas["Agencia"] = vendas["SellerG"].where(vendas["SellerG"].isin(principais), "outras")

NUMERICAS = ["Rooms", "Bathroom", "Car", "Distance", "Landsize", "BuildingArea", "YearBuilt", "Meses"]
CATEGORICAS = ["Type", "Method", "Agencia", "Regionname"]
ALVO = "ClassePreco"

vendas["Agencia"].value_counts()

Agencia
outras           3451
Nelson           1237
Jellis            966
hockingstuart     902
Barry             842
Ray               598
Buxton            507
Marshall          478
Biggin            332
Brad              291
Woodards          252
Fletchers         232
Jas               201
Sweeney           183
Greg              180
McGrath           165
Name: count, dtype: int64

Separamos 80% das vendas para treino e 20% para teste, com a mesma proporção de cada classe nos dois.

In [4]:
treino, teste = train_test_split(vendas, test_size=0.2, stratify=vendas[ALVO], random_state=SEMENTE)
treino, teste = treino.reset_index(drop=True), teste.reset_index(drop=True)
y_treino, y_teste = treino[ALVO].to_numpy(), teste[ALVO].to_numpy()

print(f"Treino: {len(treino)} | Teste: {len(teste)}")
pd.DataFrame({"treino": treino[ALVO].value_counts(normalize=True),
              "teste": teste[ALVO].value_counts(normalize=True)}).round(3)

Treino: 8653 | Teste: 2164


,treino,teste
ClassePreco,,
normal,0.523,0.523
alto,0.247,0.247
baixo,0.230,0.230


A árvore só aceita números e não pode haver valores em falta. Quando falta `BuildingArea` usamos a mediana das vendas com o mesmo `Type` e `Rooms`, quando falta `YearBuilt` a mediana do mesmo `Type` e quando falta `Car` a mediana de todas as vendas. As medianas vêm do conjunto de treino. As colunas `AreaEmFalta` e `AnoEmFalta` indicam as vendas em que a área ou o ano estavam em falta. Cada variável categórica passa a uma coluna de 0 e 1 por categoria.

In [5]:
def preparar(treino, outro):
    """Preencher os valores em falta com medianas do treino e passar as categóricas a colunas de 0 e 1."""
    mediana_area = treino.groupby(["Type", "Rooms"])["BuildingArea"].median()
    mediana_area_tipo = treino.groupby("Type")["BuildingArea"].median()
    mediana_ano = treino.groupby("Type")["YearBuilt"].median()
    mediana_car = treino["Car"].median()

    def transformar(df):
        df = df.copy()
        df["AreaEmFalta"] = df["BuildingArea"].isna().astype(int)
        df["AnoEmFalta"] = df["YearBuilt"].isna().astype(int)

        grupo = pd.MultiIndex.from_frame(df[["Type", "Rooms"]])
        area = pd.Series(mediana_area.reindex(grupo).to_numpy(), index=df.index)
        area = area.fillna(df["Type"].map(mediana_area_tipo))
        df["BuildingArea"] = df["BuildingArea"].fillna(area)
        df["YearBuilt"] = df["YearBuilt"].fillna(df["Type"].map(mediana_ano))
        df["Car"] = df["Car"].fillna(mediana_car)
        df["AreaPorDivisao"] = df["BuildingArea"] / df["Rooms"]

        numericas = df[NUMERICAS + ["AreaPorDivisao", "AreaEmFalta", "AnoEmFalta"]]
        return pd.concat([numericas, pd.get_dummies(df[CATEGORICAS], dtype=int)], axis=1)

    X_treino = transformar(treino)
    # Pôr a 0 as colunas das categorias que não aparecem em `outro`
    X_outro = transformar(outro).reindex(columns=X_treino.columns, fill_value=0)
    return X_treino, X_outro


X_treino, X_teste = preparar(treino, teste)
print(f"{X_treino.shape[1]} variáveis | valores em falta no treino: {X_treino.isna().sum().sum()}, no teste: {X_teste.isna().sum().sum()}")
X_treino.head()

41 variáveis | faltas no treino: 0, no teste: 0


,Rooms,Bathroom,Car,Distance,Landsize,BuildingArea,YearBuilt,Meses,AreaPorDivisao,AreaEmFalta,AnoEmFalta,Type_h,Type_t,Type_u,Method_S,Method_SA,Method_SP,Agencia_Barry,Agencia_Biggin,Agencia_Brad,Agencia_Buxton,Agencia_Fletchers,Agencia_Greg,Agencia_Jas,Agencia_Jellis,Agencia_Marshall,Agencia_McGrath,Agencia_Nelson,Agencia_Ray,Agencia_Sweeney,Agencia_Woodards,Agencia_hockingstuart,Agencia_outras,Regionname_Eastern Metropolitan,Regionname_Eastern Victoria,Regionname_Northern Metropolitan,Regionname_Northern Victoria,Regionname_South-Eastern Metropolitan,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria
0,3,1.0,1.0,12.8,554.0,147.0,1960.0,4.960578,49.000000,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
1,2,1.0,2.0,8.4,482.0,96.0,1960.0,14.783180,48.000000,1,1,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,2,1.0,1.0,7.0,120.0,70.0,1975.0,15.932983,35.000000,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4,1.0,2.0,14.0,604.0,183.0,1965.0,4.960578,45.750000,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
4,3,1.0,2.0,11.0,211.0,115.0,1975.0,18.462549,38.333333,1,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0


## Treinar a árvore

Estamos a usar o código da árvore em `libs/decisiontrees.py`. Em cada nó escolhe a pergunta `x <= valor` que melhor separa as classes, medida com Gini ou entropia, e para de dividir com `max_depth`, `min_samples_split`, `min_samples_leaf` ou `min_impurity_decrease`. Para a primeira árvore usamos Gini, profundidade máxima 6 para a árvore se conseguir ler, pelo menos 30 vendas por folha para as percentagens de cada folha não virem de poucas vendas, e `class_weight="balanced"`, para garantir que há o mesmo peso nas três classes, já que cerca de metade das vendas são `normal`.

In [6]:
PARAMS_INICIAIS = dict(criterion="gini", max_depth=6, min_samples_leaf=30, class_weight="balanced")

arvore = DecisionTreeClassifier(**PARAMS_INICIAIS).fit(X_treino, y_treino)
print(f"Profundidade: {arvore.get_depth()} | folhas: {arvore.get_n_leaves()}\n")
print(arvore.export_text(max_depth=3))

Profundidade: 6 | folhas: 42

AreaPorDivisao <= 50.5
|   AreaPorDivisao <= 38.0
|   |   YearBuilt <= 1939.5
|   |   |   → normal  (n=253; alto: 22%, baixo: 23%, normal: 55%)  [...]
|   |   YearBuilt > 1939.5
|   |   |   → baixo  (n=1008; alto: 10%, baixo: 39%, normal: 51%)  [...]
|   AreaPorDivisao > 38.0
|   |   YearBuilt <= 1935.5
|   |   |   → alto  (n=588; alto: 36%, baixo: 14%, normal: 50%)  [...]
|   |   YearBuilt > 1935.5
|   |   |   → baixo  (n=5456; alto: 22%, baixo: 24%, normal: 54%)  [...]
AreaPorDivisao > 50.5
|   YearBuilt <= 1925.5
|   |   BuildingArea <= 167.5
|   |   |   → alto  (n=102; alto: 47%, baixo: 8%, normal: 45%)  [...]
|   |   BuildingArea > 167.5
|   |   |   → alto  (n=143; alto: 71%, baixo: 1%, normal: 27%)  [...]
|   YearBuilt > 1925.5
|   |   Type_u = 0
|   |   |   → alto  (n=960; alto: 33%, baixo: 14%, normal: 53%)  [...]
|   |   Type_u = 1
|   |   |   → alto  (n=143; alto: 59%, baixo: 6%, normal: 35%)  [...]


In [7]:
# Comparar com a árvore do scikit-learn com os mesmos parâmetros
from sklearn.tree import DecisionTreeClassifier as ArvoreSklearn

referencia = ArvoreSklearn(**PARAMS_INICIAIS, random_state=SEMENTE).fit(X_treino, y_treino)
iguais = (referencia.predict(X_treino) == arvore.predict(X_treino)).mean()
print(f"Previsões iguais às do scikit-learn: {iguais:.1%}")

Previsões iguais às do scikit-learn: 100.0%


## Avaliar

Validação cruzada com 5 folds no treino. Cerca de metade das vendas são `normal`, por isso usamos o F1 macro em vez da accuracy. Comparamos prever sempre `normal`, uma árvore só com as variáveis da regressão e uma árvore com todas.

In [8]:
VARIAVEIS_REGRESSAO = ["Rooms", "Bathroom", "Car", "Distance", "Landsize", "Meses", "Type_h", "Type_t", "Type_u"]


def validacao_cruzada(params, colunas=None, folds=5):
    """Prever cada venda do treino com a árvore treinada nos outros folds."""
    previsoes = np.empty(len(treino), dtype=object)
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEMENTE)
    for idx_tr, idx_val in cv.split(treino, y_treino):
        X_tr, X_val = preparar(treino.iloc[idx_tr], treino.iloc[idx_val])
        if colunas is not None:
            X_tr, X_val = X_tr[colunas], X_val[colunas]
        modelo = DecisionTreeClassifier(**params).fit(X_tr, y_treino[idx_tr])
        previsoes[idx_val] = modelo.predict(X_val)
    return previsoes


def metricas(y, previsto):
    recalls = recall_score(y, previsto, labels=CLASSES, average=None, zero_division=0)
    return {"accuracy": accuracy_score(y, previsto),
            "f1_macro": f1_score(y, previsto, labels=CLASSES, average="macro", zero_division=0),
            **{f"recall_{c}": r for c, r in zip(CLASSES, recalls)}}


prev_regressao = validacao_cruzada(PARAMS_INICIAIS, VARIAVEIS_REGRESSAO)
prev_todas = validacao_cruzada(PARAMS_INICIAIS)

pd.DataFrame({
    "Sempre 'normal'": metricas(y_treino, np.full(len(y_treino), "normal")),
    "Árvore: variáveis da regressão": metricas(y_treino, prev_regressao),
    "Árvore: todas as variáveis": metricas(y_treino, prev_todas),
}).T.round(3)

,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Sempre 'normal',0.523,0.229,0.000,1.000,0.000
Árvore: variáveis da regressão,0.394,0.390,0.490,0.331,0.437
Árvore: todas as variáveis,0.423,0.420,0.475,0.351,0.527


Em F1 macro, prever sempre `normal` fica em 0,23. A árvore só com as variáveis da regressão chega a 0,39, por isso a regressão ainda não apanha todo o efeito dessas variáveis. Com as restantes variáveis sobe para 0,42. A accuracy fica abaixo de 52% porque os pesos por classe fazem a árvore prever mais vezes `baixo` e `alto`.

In [9]:
pd.DataFrame(confusion_matrix(y_treino, prev_todas, labels=CLASSES),
             index=[f"real {c}" for c in CLASSES],
             columns=[f"previsto {c}" for c in CLASSES])

,previsto baixo,previsto normal,previsto alto
real baixo,945,538,508
real normal,1431,1588,1503
real alto,486,527,1127


Cerca de um quarto das vendas `baixo` são previstas como `alto` e 23% das `alto` como `baixo`. A maioria dos erros são vendas `normal` previstas como `baixo` ou `alto`.

In [10]:
arvore.feature_importances().head(12).round(3)

AreaPorDivisao                      0.387
YearBuilt                           0.171
Landsize                            0.136
Bathroom                            0.083
Distance                            0.077
BuildingArea                        0.053
Regionname_Southern Metropolitan    0.041
Type_u                              0.029
Agencia_Marshall                    0.010
Meses                               0.009
Car                                 0.005
Agencia_Sweeney                     0.000
dtype: float64

In [11]:
arvore.rules().head(10)

,regra,classe,n,pureza,%alto,%baixo,%normal
0,38.0 < AreaPorDivisao <= 50.5 E YearBuilt > 1935.5 E Distance <= 11.6 E 339.5 < Landsize <= 709.5,baixo,1141,0.317,0.143,0.317,0.540
1,38.0 < AreaPorDivisao <= 50.5 E YearBuilt > 1935.5 E Distance <= 11.6 E Landsize <= 339.5 E Regionname_Southern Metropolitan = 0,alto,1039,0.297,0.297,0.195,0.507
2,38.0 < AreaPorDivisao <= 50.5 E YearBuilt > 1935.5 E Distance > 11.6 E Bathroom > 1.5 E Landsize <= 1057.5,normal,1018,0.625,0.132,0.244,0.625
3,38.0 < AreaPorDivisao <= 50.5 E YearBuilt > 1935.5 E Distance > 11.6 E Bathroom <= 1.5 E Landsize <= 701.5,normal,871,0.626,0.224,0.150,0.626
4,38.0 < AreaPorDivisao <= 50.5 E YearBuilt > 1935.5 E Distance <= 11.6 E Landsize <= 339.5 E Regionname_Southern Metropolitan = 1,baixo,695,0.306,0.236,0.306,0.458
5,50.5 < AreaPorDivisao <= 68.9 E 1925.5 < YearBuilt <= 2012.5 E Type_u = 0 E Distance > 6.3,normal,552,0.612,0.255,0.132,0.612
6,38.0 < AreaPorDivisao <= 50.5 E YearBuilt > 1935.5 E Distance <= 11.6 E Landsize > 709.5,alto,455,0.321,0.321,0.268,0.411
7,38.0 < AreaPorDivisao <= 50.5 E YearBuilt <= 1935.5 E Agencia_Marshall = 0 E Landsize > 100.0 E BuildingArea > 96.5,alto,370,0.359,0.359,0.135,0.505
8,AreaPorDivisao <= 38.0 E 1939.5 < YearBuilt <= 1971.5 E Distance <= 11.6 E Regionname_Southern Metropolitan = 0,baixo,255,0.463,0.090,0.463,0.447
9,AreaPorDivisao <= 38.0 E YearBuilt > 1939.5 E Distance > 11.6 E Bathroom <= 1.5 E BuildingArea > 87.5,normal,211,0.716,0.123,0.161,0.716


In [12]:
# Folhas que preveem baixo ou alto, das mais puras para as menos puras
regras = arvore.rules()
fora_do_normal = regras[(regras["classe"] != "normal") & (regras["n"] >= 30)]
fora_do_normal.sort_values("pureza", ascending=False).head(8)

,regra,classe,n,pureza,%alto,%baixo,%normal
41,50.5 < AreaPorDivisao <= 61.2 E YearBuilt > 1925.5 E Type_u = 1 E BuildingArea > 57.5 E Meses <= 13.1,alto,30,0.933,0.933,0.033,0.033
37,AreaPorDivisao > 50.5 E YearBuilt <= 1925.5 E BuildingArea > 231.0 E Landsize > 638.0,alto,31,0.903,0.903,0.000,0.097
31,AreaPorDivisao > 50.5 E YearBuilt <= 1925.5 E 167.5 < BuildingArea <= 231.0 E Regionname_Southern Metropolitan = 0,alto,36,0.778,0.778,0.000,0.222
36,AreaPorDivisao > 50.5 E YearBuilt <= 1925.5 E BuildingArea > 231.0 E Landsize <= 638.0,alto,31,0.774,0.774,0.065,0.161
23,AreaPorDivisao > 68.9 E YearBuilt > 1989.0 E Type_u = 0 E Regionname_Southern Metropolitan = 1,alto,52,0.712,0.712,0.000,0.288
39,38.0 < AreaPorDivisao <= 50.5 E YearBuilt <= 1935.5 E Agencia_Marshall = 1 E BuildingArea > 135.5,alto,30,0.700,0.700,0.100,0.200
30,50.5 < AreaPorDivisao <= 61.2 E YearBuilt > 1925.5 E Type_u = 1 E BuildingArea > 57.5 E Meses > 13.1,alto,37,0.676,0.676,0.081,0.243
40,38.0 < AreaPorDivisao <= 50.5 E YearBuilt <= 1935.5 E Agencia_Marshall = 0 E Landsize <= 100.0,alto,30,0.667,0.667,0.100,0.233


A área por divisão é a variável mais importante, seguida do ano de construção. As vendas com divisões grandes tendem a ficar acima do esperado e as com divisões pequenas abaixo. Os imóveis construídos até 1925 com divisões grandes, mais de 231 m² de área e mais de 638 m² de terreno têm 90% de vendas `alto`. As unidades com 50 a 61 m² por divisão também ficam quase sempre acima. A Marshall aparece numa regra e o mês da venda quase não conta, porque já está na regressão.

## Otimizar os hiperparâmetros

Testamos todas as combinações de `criterion`, `class_weight`, `max_depth`, `min_samples_leaf` e `min_samples_split` com a mesma validação cruzada. Em `class_weight` comparamos `balanced` com a raiz quadrada desses pesos, que fica a meio caminho entre não usar pesos e `balanced`. Para cada combinação guardamos o F1 macro médio no treino e na validação. Os folds são preparados antes da pesquisa e reutilizados em todas as combinações.

In [13]:
from itertools import product
from time import perf_counter

# Pesos balanced: n / (3 * n_classe). Os pesos raiz são a raiz quadrada destes
frequencia = pd.Series(y_treino).value_counts(normalize=True)
PESOS = {"balanced": "balanced", "raiz": (1 / (3 * frequencia) ** 0.5).round(3).to_dict()}
print("Pesos raiz:", PESOS["raiz"])

GRELHA = {
    "criterion": ["gini", "entropy"],
    "class_weight": ["balanced", "raiz"],
    "max_depth": [6, 8, 10, 12],
    "min_samples_leaf": [10, 20, 40, 80],
    "min_samples_split": [2, 150],
}

# Preparar os 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
folds = []
for idx_tr, idx_val in cv.split(treino, y_treino):
    X_tr, X_val = preparar(treino.iloc[idx_tr], treino.iloc[idx_val])
    folds.append((X_tr, y_treino[idx_tr], X_val, y_treino[idx_val]))

resultados = []
inicio = perf_counter()
for valores in product(*GRELHA.values()):
    combinacao = dict(zip(GRELHA, valores))
    params = {**combinacao, "class_weight": PESOS[combinacao["class_weight"]]}
    f1_tr, f1_val = [], []
    for X_tr, y_tr, X_val, y_val in folds:
        modelo = DecisionTreeClassifier(**params).fit(X_tr, y_tr)
        f1_tr.append(f1_score(y_tr, modelo.predict(X_tr), average="macro"))
        f1_val.append(f1_score(y_val, modelo.predict(X_val), average="macro"))
    resultados.append({**combinacao,
                       "f1_treino": np.mean(f1_tr), "f1_validacao": np.mean(f1_val), "desvio_validacao": np.std(f1_val)})

resultados = pd.DataFrame(resultados).sort_values("f1_validacao", ascending=False, ignore_index=True)
print(f"{len(resultados)} combinações em {perf_counter() - inicio:.0f} s")
resultados.head(10).round(3)

Pesos raiz: {'normal': 0.799, 'alto': 1.161, 'baixo': 1.204}


128 combinações em 182 s


,criterion,class_weight,max_depth,min_samples_leaf,min_samples_split,f1_treino,f1_validacao,desvio_validacao
0,gini,raiz,12,10,150,0.526,0.453,0.014
1,gini,raiz,12,20,150,0.521,0.451,0.012
2,gini,raiz,10,20,150,0.514,0.451,0.015
3,gini,raiz,10,10,150,0.516,0.451,0.018
4,gini,raiz,8,40,150,0.493,0.450,0.014
5,gini,raiz,10,40,150,0.508,0.450,0.011
6,gini,raiz,8,20,150,0.495,0.450,0.016
7,gini,raiz,8,40,2,0.499,0.449,0.011
8,gini,raiz,8,80,2,0.485,0.449,0.014
9,gini,raiz,8,80,150,0.485,0.449,0.014


Melhor combinação para cada profundidade, com o F1 no treino ao lado do F1 na validação.

In [14]:
por_profundidade = resultados.loc[resultados.groupby("max_depth")["f1_validacao"].idxmax()]
por_profundidade[["max_depth", "criterion", "class_weight", "min_samples_leaf", "min_samples_split", "f1_treino", "f1_validacao", "desvio_validacao"]].round(3)

,max_depth,criterion,class_weight,min_samples_leaf,min_samples_split,f1_treino,f1_validacao,desvio_validacao
58,6,entropy,raiz,80,150,0.447,0.430,0.018
4,8,gini,raiz,40,150,0.493,0.450,0.014
2,10,gini,raiz,20,150,0.514,0.451,0.015
0,12,gini,raiz,10,150,0.526,0.453,0.014


In [15]:
# Efeito médio de cada hiperparâmetro no F1 de validação
for nome in GRELHA:
    print(resultados.groupby(nome)["f1_validacao"].mean().round(3).to_dict())

{'entropy': 0.428, 'gini': 0.432}
{'balanced': 0.424, 'raiz': 0.437}
{6: 0.42, 8: 0.431, 10: 0.436, 12: 0.434}
{10: 0.429, 20: 0.431, 40: 0.43, 80: 0.432}
{2: 0.428, 150: 0.432}


O peso das classes é o hiperparâmetro que mais conta. Os pesos raiz ficam em média 1,3 pontos acima de `balanced` e estão em todas as dez melhores combinações. A profundidade conta menos, as árvores com 10 a 12 níveis ficam cerca de 1,5 pontos acima das de 6. Gini e entropia dão quase o mesmo, tal como `min_samples_leaf` e `min_samples_split`. Nas melhores combinações a diferença entre treino e validação fica abaixo de 8 pontos.

In [16]:
MELHORES = resultados.iloc[0][list(GRELHA)].to_dict()
MELHORES["class_weight"] = PESOS[MELHORES["class_weight"]]
for nome in ["max_depth", "min_samples_leaf", "min_samples_split"]:
    MELHORES[nome] = int(MELHORES[nome])
print(MELHORES)

prev_melhor = validacao_cruzada(MELHORES)
pd.DataFrame({
    "Árvore inicial": metricas(y_treino, prev_todas),
    "Árvore otimizada": metricas(y_treino, prev_melhor),
}).T.round(3)

{'criterion': 'gini', 'class_weight': {'normal': 0.799, 'alto': 1.161, 'baixo': 1.204}, 'max_depth': 12, 'min_samples_leaf': 10, 'min_samples_split': 150}


,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Árvore inicial,0.423,0.420,0.475,0.351,0.527
Árvore otimizada,0.491,0.453,0.355,0.585,0.417


A árvore escolhida usa Gini, os pesos raiz, `max_depth=12`, `min_samples_leaf=10` e `min_samples_split=150`. Em relação à árvore inicial o F1 macro sobe de 0,42 para 0,45 e a accuracy de 0,42 para 0,49. Com pesos mais baixos nas classes `baixo` e `alto`, a árvore acerta mais vendas `normal`, 59% contra 35%, e menos `baixo` e `alto`.

## Analisar o modelo final

A árvore final é treinada com os melhores hiperparâmetros em todo o conjunto de treino.

In [17]:
final = DecisionTreeClassifier(**MELHORES).fit(X_treino, y_treino)
print(f"Profundidade: {final.get_depth()} | folhas: {final.get_n_leaves()}\n")
print(final.export_text(max_depth=3))

Profundidade: 12 | folhas: 112

AreaPorDivisao <= 50.5
|   AreaPorDivisao <= 38.0
|   |   Regionname_Southern Metropolitan = 0
|   |   |   → normal  (n=936; alto: 13%, baixo: 31%, normal: 56%)  [...]
|   |   Regionname_Southern Metropolitan = 1
|   |   |   → baixo  (n=325; alto: 11%, baixo: 49%, normal: 40%)  [...]
|   AreaPorDivisao > 38.0
|   |   Distance <= 12.1
|   |   |   → normal  (n=4036; alto: 25%, baixo: 25%, normal: 50%)  [...]
|   |   Distance > 12.1
|   |   |   → normal  (n=2008; alto: 20%, baixo: 19%, normal: 61%)  [...]
AreaPorDivisao > 50.5
|   YearBuilt <= 1929.0
|   |   BuildingArea <= 231.0
|   |   |   → alto  (n=185; alto: 53%, baixo: 5%, normal: 42%)  [...]
|   |   BuildingArea > 231.0
|   |   |   → alto  (n=66; alto: 83%, baixo: 3%, normal: 14%)
|   YearBuilt > 1929.0
|   |   Type_u = 0
|   |   |   → normal  (n=954; alto: 32%, baixo: 14%, normal: 53%)  [...]
|   |   Type_u = 1
|   |   |   → alto  (n=143; alto: 59%, baixo: 6%, normal: 35%)


In [18]:
final.feature_importances().head(12).round(3)

AreaPorDivisao                      0.242
Landsize                            0.158
YearBuilt                           0.135
Distance                            0.116
BuildingArea                        0.082
Bathroom                            0.051
Meses                               0.047
Type_u                              0.039
Regionname_Southern Metropolitan    0.038
Agencia_Marshall                    0.011
AnoEmFalta                          0.011
Rooms                               0.008
dtype: float64

In [19]:
# Folhas com pelo menos 30 vendas onde a árvore prevê baixo ou alto, das mais puras para as menos puras
regras_finais = final.rules()
regras_finais[(regras_finais["classe"] != "normal") & (regras_finais["n"] >= 30)].sort_values("pureza", ascending=False).head(10)

,regra,classe,n,pureza,%alto,%baixo,%normal
54,AreaPorDivisao > 50.5 E YearBuilt <= 1929.0 E BuildingArea > 231.0,alto,66,0.833,0.833,0.030,0.136
69,38.0 < AreaPorDivisao <= 50.5 E Distance <= 12.1 E YearBuilt <= 1911.0 E BuildingArea > 173.5,alto,40,0.725,0.725,0.050,0.225
24,AreaPorDivisao <= 38.0 E Regionname_Southern Metropolitan = 0 E Bathroom > 1.5 E 1917.5 < YearBuilt <= 2011.5 E Landsize <= 822.5 E AnoEmFalta = 0 E Regionname_South-Eastern Metropolitan = 0 E Meses > 6.0,baixo,125,0.656,0.024,0.656,0.320
31,AreaPorDivisao > 50.5 E YearBuilt <= 1929.0 E BuildingArea <= 231.0 E Car <= 1.5,alto,109,0.606,0.606,0.046,0.349
13,AreaPorDivisao > 68.9 E YearBuilt > 1989.0 E Type_u = 0,alto,141,0.589,0.589,0.085,0.326
11,AreaPorDivisao > 50.5 E YearBuilt > 1929.0 E Type_u = 1,alto,143,0.587,0.587,0.063,0.350
59,38.0 < AreaPorDivisao <= 50.5 E Distance <= 12.1 E YearBuilt <= 1911.0 E 96.5 < BuildingArea <= 173.5 E Landsize <= 206.5,alto,55,0.564,0.564,0.055,0.382
61,38.0 < AreaPorDivisao <= 50.1 E Distance <= 12.1 E YearBuilt > 1944.0 E 1.5 < Landsize <= 139.5 E Regionname_Southern Metropolitan = 1 E Meses <= 15.4 E Agencia_Marshall = 0,baixo,54,0.556,0.037,0.556,0.407
7,AreaPorDivisao <= 38.0 E Regionname_Southern Metropolitan = 1 E Rooms <= 2.5 E BuildingArea > 57.5,baixo,147,0.551,0.048,0.551,0.401
67,45.9 < AreaPorDivisao <= 50.5 E 12.1 < Distance <= 16.9 E Bathroom > 1.5 E Landsize <= 1170.0 E YearBuilt > 1935.5 E Agencia_Ray = 1,baixo,41,0.537,0.049,0.537,0.415


In [20]:
# F1 macro por tipo de imóvel, com as previsões da validação cruzada da árvore otimizada
pd.DataFrame({
    tipo: {"n": int((treino["Type"] == tipo).sum()),
           **metricas(y_treino[treino["Type"] == tipo], prev_melhor[treino["Type"] == tipo])}
    for tipo in ["h", "t", "u"]
}).T.round(3)

,n,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
h,6101.0,0.504,0.452,0.317,0.621,0.414
t,710.0,0.461,0.402,0.382,0.578,0.250
u,1842.0,0.457,0.454,0.451,0.450,0.474


A árvore final usa mais a área por divisão, o terreno, o ano de construção e a distância. Os imóveis construídos até 1929 com divisões grandes e mais de 231 m² têm 83% de vendas `alto`, e os construídos até 1911 a menos de 12 km do centro com mais de 173 m² têm 73%. As casas construídas depois de 1989 com divisões muito grandes e as unidades espaçosas ficam perto de 60% `alto`. No lado oposto, os imóveis com divisões pequenas no Southern Metropolitan têm 49% de vendas `baixo`, o dobro da média. A agência e o método de venda praticamente não contam.

As casas e as unidades têm F1 de 0,45 e as townhouses 0,40, que são também o tipo com menos vendas.

## Avaliar no teste

In [21]:
prev_teste = final.predict(X_teste)
pd.DataFrame({
    "Sempre 'normal'": metricas(y_teste, np.full(len(y_teste), "normal")),
    "Árvore final": metricas(y_teste, prev_teste),
}).T.round(3)

,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Sempre 'normal',0.523,0.229,0.000,1.000,0.000
Árvore final,0.484,0.436,0.293,0.608,0.398


In [22]:
pd.DataFrame(confusion_matrix(y_teste, prev_teste, labels=CLASSES),
             index=[f"real {c}" for c in CLASSES],
             columns=[f"previsto {c}" for c in CLASSES])

,previsto baixo,previsto normal,previsto alto
real baixo,146,277,75
real normal,216,688,227
real alto,51,271,213


No teste a árvore tem F1 macro de 0,44, um pouco abaixo dos 0,45 da validação cruzada. Fica acima de prever sempre `normal`, que tem 0,23, com accuracy próxima, 0,48 contra 0,52. A árvore confunde pouco as classes extremas, só 15% das vendas `baixo` são previstas como `alto` e 10% das `alto` como `baixo`. A maior parte dos erros são vendas `baixo` e `alto` previstas como `normal`.